# SpliceAI 10k Training (PyTorch, H100 friendly)

이 노트북은 로컬에서 만든 HDF5 dataset(`train_*.h5`, `val_*.h5`, `test_*.h5`)을 이용해  
`spliceai-pytorch`의 **SpliceAI 10k 모델**을 학습합니다.

- 입력: `X` (uint8 code; A=0,C=1,G=2,T=3,N=4) shape `[N, 15000]`
- 라벨: `Y` (uint8 class index; 0/1/2) shape `[N, 5000]`
- 모델 입력 텐서: `[B, 4, 15000]`
- 모델 출력 텐서: `[B, 5000, 3]`

> RAM/디스크 절약을 위해 one-hot을 HDF5에 저장하지 않습니다. 학습 시 on-the-fly one-hot 변환합니다.


In [ ]:
# (Colab) Install deps
!pip -q install spliceai-pytorch h5py tqdm scikit-learn


In [ ]:
import os, glob, json, math, random
import numpy as np
import h5py
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.metrics import average_precision_score
from spliceai_pytorch import SpliceAI

print("torch:", torch.__version__)
print("cuda:", torch.version.cuda)
print("device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")


## 1) Dataset 위치 설정

- `DATA_DIR`에는 `train_*.h5`, `val_*.h5`, `test_*.h5`, `dataset_stats.json`가 있어야 합니다.
- Google Drive에 올려둔 뒤 `/content`로 복사해두면 훨씬 빠릅니다.


In [ ]:
DATA_DIR = "/content/out_spliceai10k_ds"  # TODO: set me
STATS_PATH = os.path.join(DATA_DIR, "dataset_stats.json")

with open(STATS_PATH, "r") as f:
    stats = json.load(f)
stats


In [ ]:
# Collect shard files
train_files = sorted(glob.glob(os.path.join(DATA_DIR, "train_*.h5")))
val_files   = sorted(glob.glob(os.path.join(DATA_DIR, "val_*.h5")))
test_files  = sorted(glob.glob(os.path.join(DATA_DIR, "test_*.h5")))

print("train shards:", len(train_files))
print("val shards:", len(val_files))
print("test shards:", len(test_files))
print("example:", train_files[:2])


## 2) HDF5 Dataset Loader (low RAM)

- `X`는 uint8 code이므로, GPU에서 embedding lookup으로 one-hot을 만듭니다.
- HDF5는 멀티워커에서 file handle 문제가 생길 수 있어, 기본은 `num_workers=0`로 시작하세요.
  - 속도가 부족하면 `num_workers=2` 정도로 올리되, 문제 생기면 다시 0으로.


In [ ]:
from torch.utils.data import Dataset, DataLoader

BASE_EMBED = torch.tensor([
    [1,0,0,0],  # A
    [0,1,0,0],  # C
    [0,0,1,0],  # G
    [0,0,0,1],  # T
    [0,0,0,0],  # N/other
], dtype=torch.float32)

class H5ShardDataset(Dataset):
    def __init__(self, h5_path: str):
        self.h5_path = h5_path
        self._h5 = None
        self._len = None

    def _ensure_open(self):
        if self._h5 is None:
            self._h5 = h5py.File(self.h5_path, "r")
            self._len = self._h5["X"].shape[0]

    def __len__(self):
        self._ensure_open()
        return int(self._len)

    def __getitem__(self, idx: int):
        self._ensure_open()
        x = self._h5["X"][idx]  # uint8 [15000]
        y = self._h5["Y"][idx]  # uint8 [5000]
        return x, y

    def close(self):
        if self._h5 is not None:
            self._h5.close()
            self._h5 = None

class MultiShardDataset(Dataset):
    """Concatenate multiple shard files with cumulative index mapping."""
    def __init__(self, files: list[str]):
        self.files = files
        self.datasets = [H5ShardDataset(p) for p in files]
        self.cum = []
        s = 0
        for ds in self.datasets:
            n = len(ds)
            s += n
            self.cum.append(s)

    def __len__(self):
        return self.cum[-1] if self.cum else 0

    def _locate(self, idx: int):
        # binary search over cum
        lo, hi = 0, len(self.cum)-1
        while lo <= hi:
            mid = (lo+hi)//2
            if idx < self.cum[mid]:
                hi = mid-1
            else:
                lo = mid+1
        file_idx = lo
        prev = self.cum[file_idx-1] if file_idx > 0 else 0
        local_idx = idx - prev
        return file_idx, local_idx

    def __getitem__(self, idx: int):
        fi, li = self._locate(idx)
        return self.datasets[fi][li]

    def close(self):
        for ds in self.datasets:
            ds.close()

def collate_uint8(batch):
    xs, ys = zip(*batch)
    x = torch.from_numpy(np.stack(xs, axis=0)).to(torch.uint8)  # [B,15000]
    y = torch.from_numpy(np.stack(ys, axis=0)).to(torch.uint8)  # [B,5000]
    return x, y


In [ ]:
# Create loaders
BATCH_SIZE = 8  # TODO adjust for H100 (try 16, 32 if VRAM allows)
NUM_WORKERS = 0 # start safe; try 2 later if stable & you want speed

train_ds = MultiShardDataset(train_files)
val_ds   = MultiShardDataset(val_files)
test_ds  = MultiShardDataset(test_files)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True, collate_fn=collate_uint8, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True, collate_fn=collate_uint8)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=True, collate_fn=collate_uint8)

len(train_ds), len(val_ds), len(test_ds)


## 3) Model / Loss / Optimizer

- `spliceai-pytorch`는 `SpliceAI.from_preconfigured('10k')`로 모델을 만들 수 있습니다.
- Loss는 기본 `CrossEntropyLoss` (class index target).
- 클래스 불균형이 심하면 class weight를 줄 수도 있습니다.


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = SpliceAI.from_preconfigured("10k").to(device)

# optional class weights from dataset_stats (rough; you can tune)
label_counts = np.array(stats["label_counts"]["train"], dtype=np.float64)
label_counts = np.maximum(label_counts, 1.0)
inv = label_counts.sum() / (3.0 * label_counts)
class_weight = torch.tensor(inv, dtype=torch.float32, device=device)

print("label_counts(train) =", label_counts)
print("class_weight =", class_weight)

criterion = nn.CrossEntropyLoss(weight=class_weight)
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-2)
scheduler = optim.lr_scheduler.MultiStepLR(optimizer, milestones=[6,7,8,9], gamma=0.5)

# AMP
use_amp = True
scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

# quick shape check
x0, y0 = next(iter(train_loader))
codes = x0.to(device, non_blocking=True).long()  # [B,15000]
x_oh = BASE_EMBED.to(device)[codes].permute(0,2,1).contiguous()  # [B,4,15000]
with torch.no_grad():
    out = model(x_oh)
print("out:", out.shape, "y:", y0.shape)


## 4) Training / Eval loop

- 메트릭은 **빠른 확인용**으로 batch 일부만 사용합니다.
- 전체 AUPRC는 시간이 오래 걸릴 수 있어서, 필요하면 epoch 끝에서 샘플 수를 늘려 계산하세요.


In [ ]:
def topk_accuracy_flat(pred_prob: torch.Tensor, y_true_bin: torch.Tensor) -> float:
    """SpliceAI style top-k accuracy on flattened arrays."""
    pred = pred_prob.reshape(-1)
    y = y_true_bin.reshape(-1)
    k = int(y.sum().item())
    if k == 0:
        return float('nan')
    topk = torch.topk(pred, k).indices
    return float((y[topk] == 1).float().mean().item())

@torch.no_grad()
def evaluate(loader, max_batches: int = 200):
    model.eval()
    losses = []
    acc_topk = []
    don_topk = []
    ys_acc, ps_acc, ys_don, ps_don = [], [], [], []

    for bi, (x_codes, y_codes) in enumerate(tqdm(loader, total=min(len(loader), max_batches), leave=False)):
        if bi >= max_batches:
            break
        x_codes = x_codes.to(device, non_blocking=True).long()
        y_codes = y_codes.to(device, non_blocking=True).long()  # [B,5000]

        x_oh = BASE_EMBED.to(device)[x_codes].permute(0,2,1).contiguous()  # [B,4,15000]
        out = model(x_oh)  # [B,5000,3]
        loss = criterion(out.reshape(-1,3), y_codes.reshape(-1))
        losses.append(loss.item())

        prob = out.softmax(dim=-1)
        y_acc = (y_codes == 1).float()
        y_don = (y_codes == 2).float()
        acc_topk.append(topk_accuracy_flat(prob[:,:,1], y_acc))
        don_topk.append(topk_accuracy_flat(prob[:,:,2], y_don))

        ys_acc.append(y_acc.detach().cpu().numpy().reshape(-1))
        ps_acc.append(prob[:,:,1].detach().cpu().numpy().reshape(-1))
        ys_don.append(y_don.detach().cpu().numpy().reshape(-1))
        ps_don.append(prob[:,:,2].detach().cpu().numpy().reshape(-1))

    loss_mean = float(np.mean(losses)) if losses else float('nan')
    acc_topk_mean = float(np.nanmean(acc_topk)) if acc_topk else float('nan')
    don_topk_mean = float(np.nanmean(don_topk)) if don_topk else float('nan')

    y_acc_all = np.concatenate(ys_acc) if ys_acc else np.array([])
    p_acc_all = np.concatenate(ps_acc) if ps_acc else np.array([])
    y_don_all = np.concatenate(ys_don) if ys_don else np.array([])
    p_don_all = np.concatenate(ps_don) if ps_don else np.array([])

    auprc_acc = float(average_precision_score(y_acc_all, p_acc_all)) if y_acc_all.size else float('nan')
    auprc_don = float(average_precision_score(y_don_all, p_don_all)) if y_don_all.size else float('nan')

    return {
        'loss': loss_mean,
        'topk_acceptor': acc_topk_mean,
        'topk_donor': don_topk_mean,
        'auprc_acceptor': auprc_acc,
        'auprc_donor': auprc_don,
        'n_batches': len(losses),
    }

def train_one_epoch(max_batches: int | None = None):
    model.train()
    losses = []
    for bi, (x_codes, y_codes) in enumerate(tqdm(train_loader, total=(max_batches or len(train_loader)))):
        if max_batches is not None and bi >= max_batches:
            break

        x_codes = x_codes.to(device, non_blocking=True).long()
        y_codes = y_codes.to(device, non_blocking=True).long()

        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=use_amp):
            x_oh = BASE_EMBED.to(device)[x_codes].permute(0,2,1).contiguous()
            out = model(x_oh)
            loss = criterion(out.reshape(-1,3), y_codes.reshape(-1))

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        losses.append(loss.item())

    return float(np.mean(losses)) if losses else float('nan')


In [ ]:
EPOCHS = 10
MAX_TRAIN_BATCHES = None  # set small number for debug, e.g. 200
EVAL_BATCHES = 200

SAVE_DIR = "/content"
CKPT_PATH = os.path.join(SAVE_DIR, "spliceai10k_state_dict.pt")

history = []
for epoch in range(1, EPOCHS+1):
    train_loss = train_one_epoch(max_batches=MAX_TRAIN_BATCHES)
    val_metrics = evaluate(val_loader, max_batches=EVAL_BATCHES)
    scheduler.step()

    row = {"epoch": epoch, "train_loss": train_loss, **{f"val_{k}": v for k,v in val_metrics.items()}}
    history.append(row)
    print(row)

    torch.save(model.state_dict(), CKPT_PATH)

print("Saved:", CKPT_PATH)


## 5) Test set 평가 (선택)

epoch 끝에서 test set metric을 확인하고 싶으면 실행하세요.


In [ ]:
test_metrics = evaluate(test_loader, max_batches=500)
test_metrics


### 마무리

- `spliceai10k_state_dict.pt`를 다운로드하거나 Drive에 저장해두세요.
- 다음 단계(variant scoring / Step3 backend)로 붙일 때는:
  - 모델 입력: 4x(5000+10000)=4x15000 one-hot
  - 모델 출력: 5000x3 softmax
  - Mission6에서 했던 scoreing 로직(Δ확률 등)을 여기에 얹으면 됩니다.
